In [33]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
    .master("local[4]")
    .appName("tp")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.host", "localhost")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate())

print(spark.sparkContext.uiWebUrl)

http://localhost:4040


In [4]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
    .master("local[4]")                            # 1 machine, 4 cores = 4 task slots
    .appName("tp")
    .config("spark.sql.shuffle.partitions", "8")   # 200 is too much for a laptop
    .getOrCreate())

# Customers: 1000 rows
customers = spark.range(1, 1001).select(
    F.col("id").alias("cust_id"),
    F.concat(F.lit("name_"), F.col("id")).alias("name"),
    F.element_at(F.array(F.lit("MA"), F.lit("FR"), F.lit("US"), F.lit("DE")),
                 (F.col("id") % 4 + 1).cast("int")).alias("country"))

# Orders: 1,000,000 rows, and half of them belong to customer 1 (a hot key, for the skew lab)
orders = spark.range(1_000_000).select(
    F.col("id").alias("order_id"),
    F.when(F.rand(42) < 0.5, F.lit(1))
     .otherwise((F.rand(7) * 1000).cast("int") + 1).alias("cust_id"),
    (F.rand(1) * 500).alias("amount"))

# Employees: 5000 rows, 10 departments
employees = spark.range(5000).select(
    F.col("id").alias("emp_id"),
    F.concat(F.lit("emp_"), F.col("id")).alias("name"),
    (F.col("id") % 10).alias("dept_id"),
    (F.rand(3) * 8000 + 2000).cast("int").alias("salary"),
    F.date_add(F.lit("2015-01-01"), (F.rand(5) * 3000).cast("int")).alias("hire_date"))

# Save as Parquet so every lab reads the same data
customers.write.mode("overwrite").parquet("data/customers")
orders.write.mode("overwrite").parquet("data/orders")
employees.write.mode("overwrite").parquet("data/employees")

In [3]:
orders = spark.read.parquet("data/orders")
print(orders.count())
print(orders.rdd.getNumPartitions())

1000000
4


In [4]:
orders = spark.read.parquet("data/orders")            # (a)
big = orders.filter(F.col("amount") > 400)            # (b)
agg = big.groupBy("cust_id").count()                  # (c)
agg.show(5)                                           # (d)
agg.count()                                           # (e)

+-------+-----+
|cust_id|count|
+-------+-----+
|    171|   98|
|     67|  113|
|    536|   97|
|    734|   94|
|    107|   96|
+-------+-----+
only showing top 5 rows


1000

In [7]:
spark.sparkContext.setJobDescription("LAB1 show(5)")
agg.show(5)

spark.sparkContext.setJobDescription("LAB1 count")
agg.count()

+-------+-----+
|cust_id|count|
+-------+-----+
|    171|   98|
|     67|  113|
|    536|   97|
|    734|   94|
|    107|   96|
+-------+-----+
only showing top 5 rows


1000

In [8]:
big = spark.read.parquet("data/orders").filter(F.col("amount") > 400).select("cust_id", "amount")
big.explain(True)

== Parsed Logical Plan ==
'Project ['cust_id, 'amount]
+- Filter (amount#75 > cast(400 as double))
   +- Relation [order_id#73L,cust_id#74,amount#75] parquet

== Analyzed Logical Plan ==
cust_id: int, amount: double
Project [cust_id#74, amount#75]
+- Filter (amount#75 > cast(400 as double))
   +- Relation [order_id#73L,cust_id#74,amount#75] parquet

== Optimized Logical Plan ==
Project [cust_id#74, amount#75]
+- Filter (isnotnull(amount#75) AND (amount#75 > 400.0))
   +- Relation [order_id#73L,cust_id#74,amount#75] parquet

== Physical Plan ==
*(1) Filter (isnotnull(amount#75) AND (amount#75 > 400.0))
+- *(1) ColumnarToRow
   +- FileScan parquet [cust_id#74,amount#75] Batched: true, DataFilters: [isnotnull(amount#75), (amount#75 > 400.0)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/oualid/Desktop/spark/data/orders], PartitionFilters: [], PushedFilters: [IsNotNull(amount), GreaterThan(amount,400.0)], ReadSchema: struct<cust_id:int,amount:double>

#
# A fatal error 

In [5]:
print(orders.rdd.getNumPartitions())                    # orders → 4
print(orders.repartition(10).rdd.getNumPartitions())    # orders → repartition(10)
print(orders.coalesce(2).rdd.getNumPartitions())        # orders → coalesce(2)
print(orders.coalesce(10).rdd.getNumPartitions())       # orders → coalesce(10)

4


[Stage 14:>                                                         (0 + 4) / 4]

10
2
4


In [6]:
orders = spark.read.parquet("data/orders")
customers = spark.read.parquet("data/customers")

j1 = orders.join(customers, "cust_id")
j1.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [cust_id#35, order_id#34L, amount#36, name#38, country#39]
   +- BroadcastHashJoin [cast(cust_id#35 as bigint)], [cust_id#37L], Inner, BuildRight, false, false
      :- Filter isnotnull(cust_id#35)
      :  +- FileScan parquet [order_id#34L,cust_id#35,amount#36] Batched: true, DataFilters: [isnotnull(cust_id#35)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/oualid/Desktop/spark/data/orders], PartitionFilters: [], PushedFilters: [IsNotNull(cust_id)], ReadSchema: struct<order_id:bigint,cust_id:int,amount:double>
      +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=299]
         +- Filter isnotnull(cust_id#37L)
            +- FileScan parquet [cust_id#37L,name#38,country#39] Batched: true, DataFilters: [isnotnull(cust_id#37L)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/oualid/Desktop/spark/data/customers], PartitionFilters: [], Pus

In [7]:
j2 = orders.join(customers.hint("merge"), "cust_id")
j2.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [cust_id#35, order_id#34L, amount#36, name#38, country#39]
   +- SortMergeJoin [cast(cust_id#35 as bigint)], [cust_id#37L], Inner
      :- Sort [cast(cust_id#35 as bigint) ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(cast(cust_id#35 as bigint), 8), ENSURE_REQUIREMENTS, [plan_id=329]
      :     +- Filter isnotnull(cust_id#35)
      :        +- FileScan parquet [order_id#34L,cust_id#35,amount#36] Batched: true, DataFilters: [isnotnull(cust_id#35)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/oualid/Desktop/spark/data/orders], PartitionFilters: [], PushedFilters: [IsNotNull(cust_id)], ReadSchema: struct<order_id:bigint,cust_id:int,amount:double>
      +- Sort [cust_id#37L ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(cust_id#37L, 8), ENSURE_REQUIREMENTS, [plan_id=330]
            +- Filter isnotnull(cust_id#37L)
               +- FileScan parquet [cust_id#37L,nam

In [8]:
spark.sparkContext.setJobDescription("LAB4 broadcast")
j1.count()

spark.sparkContext.setJobDescription("LAB4 sortmerge")
j2.count()

1000000

In [9]:
orders = spark.read.parquet("data/orders")
orders.groupBy("cust_id").count().orderBy(F.desc("count")).show(3)

[Stage 31:===========================================>              (3 + 1) / 4]

+-------+------+
|cust_id| count|
+-------+------+
|      1|500794|
|    693|   576|
|    254|   569|
+-------+------+
only showing top 3 rows


In [11]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")   # no auto broadcast

spark.sparkContext.setJobDescription("LAB5 skew join")
orders.join(customers, "cust_id").groupBy("country").count().show()
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")   # no auto broadcast

spark.sparkContext.setJobDescription("LAB5 skew join")
orders.join(customers, "cust_id").groupBy("country").count().show()

+-------+------+
|country| count|
+-------+------+
|     FR|624776|
|     DE|124537|
|     US|125005|
|     MA|125682|
+-------+------+



+-------+------+
|country| count|
+-------+------+
|     FR|624776|
|     DE|124537|
|     US|125005|
|     MA|125682|
+-------+------+



In [12]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

spark.sparkContext.setJobDescription("LAB5 AQE")
orders.join(customers, "cust_id").groupBy("country").count().show()

[Stage 62:=============================>                            (1 + 1) / 2]

+-------+------+
|country| count|
+-------+------+
|     FR|624776|
|     DE|124537|
|     US|125005|
|     MA|125682|
+-------+------+



In [14]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")

In [15]:
from pyspark.sql import Window

employees = spark.read.parquet("data/employees")

w = Window.partitionBy("dept_id").orderBy(F.col("salary").desc())

top2 = (employees
    .withColumn("rnk", F.row_number().over(w))
    .filter(F.col("rnk") <= 2))

top2.orderBy("dept_id", "rnk").show(20)
top2.explain()

+------+--------+-------+------+----------+---+
|emp_id|    name|dept_id|salary| hire_date|rnk|
+------+--------+-------+------+----------+---+
|  4290|emp_4290|      0|  9982|2022-06-30|  1|
|  2270|emp_2270|      0|  9946|2020-08-24|  2|
|  1901|emp_1901|      1|  9997|2020-11-01|  1|
|   791| emp_791|      1|  9970|2018-09-05|  2|
|  2392|emp_2392|      2|  9992|2020-12-19|  1|
|  1922|emp_1922|      2|  9990|2015-11-26|  2|
|  3713|emp_3713|      3|  9999|2017-04-16|  1|
|  4913|emp_4913|      3|  9995|2016-05-07|  2|
|  2594|emp_2594|      4|  9987|2019-11-29|  1|
|  4814|emp_4814|      4|  9985|2017-11-10|  2|
|  2275|emp_2275|      5|  9989|2017-06-27|  1|
|  3255|emp_3255|      5|  9957|2021-05-04|  2|
|  2226|emp_2226|      6|  9999|2022-07-24|  1|
|  3416|emp_3416|      6|  9997|2015-12-12|  2|
|  3437|emp_3437|      7|  9990|2022-01-15|  1|
|  4247|emp_4247|      7|  9972|2020-04-14|  2|
|  2158|emp_2158|      8|  9996|2015-08-31|  1|
|   568| emp_568|      8|  9951|2017-10-

In [16]:
customers = spark.read.parquet("data/customers")
orders = spark.read.parquet("data/orders")

sales = (orders.join(F.broadcast(customers), "cust_id")
               .select("order_id", "cust_id", "amount", "country"))

# 1) flat: no disk partitioning
sales.write.mode("overwrite").parquet("data/sales_flat")

# 2) partitioned on disk by country
sales.write.mode("overwrite").partitionBy("country").parquet("data/sales_by_country")

In [17]:
flat = spark.read.parquet("data/sales_flat")
part = spark.read.parquet("data/sales_by_country")

flat.filter("country = 'MA'").explain()   # (1)
part.filter("country = 'MA'").explain()   # (2)
part.filter("cust_id = 5").explain()      # (3)

== Physical Plan ==
*(1) Filter (isnotnull(country#185) AND (country#185 = MA))
+- *(1) ColumnarToRow
   +- FileScan parquet [order_id#182L,cust_id#183,amount#184,country#185] Batched: true, DataFilters: [isnotnull(country#185), (country#185 = MA)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/oualid/Desktop/spark/data/sales_flat], PartitionFilters: [], PushedFilters: [IsNotNull(country), EqualTo(country,MA)], ReadSchema: struct<order_id:bigint,cust_id:int,amount:double,country:string>


== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [order_id#186L,cust_id#187,amount#188,country#189] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/oualid/Desktop/spark/data/sales_by_country], PartitionFilters: [isnotnull(country#189), (country#189 = MA)], PushedFilters: [], ReadSchema: struct<order_id:bigint,cust_id:int,amount:double>


== Physical Plan ==
*(1) Filter (isnotnull(cust_id#187) AND (cust_id#187 = 5))
+- *(1) C

syntaxe

In [26]:
from pyspark.sql import SparkSession , functions as F 

spark = ( SparkSession.builder
          .master("local 4")
          .appName("my spark")
          .config("spark.sql.shuffle.partitions", "8")
          .config("spark.driver.host", "localhost")
          .config("spark.driver.bindAddress", "127.0.0.1:4041")
          .getOrCreate()
)

print(spark)

26/10/04 12:58:17 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [34]:
spark.stop()